# notebooks/02_feature_engineering.ipynb

In [1]:
import sys, os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
from data_loader import create_panel_data

## Load Panel Data

In [2]:
panel_df = create_panel_data(frequency='5_min')

print('Data Head: ')
print(panel_df.head())
print('\nData Tail: ')
print(panel_df.tail())

C:\Users\GP\MLProject\scripts\data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
C:\Users\GP\MLProject\scripts\data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
C:\Users\GP\MLProject\scripts\data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.

Data Head: 
                        rv       bpv      good       bad          rq
Date       Stock                                                    
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371
           BA     7.469396  7.016747  5.107386  2.362010  150.112379

Data Tail: 
                        rv       bpv      good       bad        rq
Date       Stock                                                  
2024-03-28 TRV    0.501900  0.450509  0.225921  0.275979  0.517683
           UNH    0.774552  0.762729  0.406755  0.367797  0.696658
           V      0.627872  0.445852  0.329977  0.297895  1.160977
           VZ     0.783853  0.706211  0.493097  0.290755  1.847938
           WMT    0.359440  0.336123  0.106481  0.252960  0.150975


C:\Users\GP\MLProject\scripts\data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)


## Integrate VIX Data

In [3]:
START_DATE = '2003-01-02'
END_DATE = '2024-03-28'

vix_path = os.path.join('..', 'data', 'VIX_History.csv')
vix_data = pd.read_csv(vix_path)

vix_data['Date'] = pd.to_datetime(vix_data['DATE'], format='%m/%d/%Y')
vix_data = vix_data.set_index('Date')
vix = vix_data[['CLOSE']].rename(columns={'CLOSE': 'vix'})
vix = vix.loc[START_DATE:END_DATE]

panel_vix = panel_df.join(vix, on='Date')
print('Panel with VIX: ')
print(panel_vix.head())

Panel with VIX: 
                        rv       bpv      good       bad          rq    vix
Date       Stock                                                           
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402  25.39
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213  25.39
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309  25.39
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371  25.39
           BA     7.469396  7.016747  5.107386  2.362010  150.112379  25.39


## Define Target Variable (Y_reg)

In [4]:
df = panel_vix.sort_index()

df['Y_reg'] = df.groupby('Stock')['rv'].shift(-1)

print('Example for AAPL: ')
display(df.loc[pd.IndexSlice[:, 'AAPL'], :].tail()[['rv', 'Y_reg']])

Example for AAPL: 


,,rv,Y_reg
Date,Stock,,
2024-03-22,AAPL,1.216548,0.682342
2024-03-25,AAPL,0.682342,0.425990
2024-03-26,AAPL,0.425990,0.959378
2024-03-27,AAPL,0.959378,0.563307
2024-03-28,AAPL,0.563307,NaN


## Engineer Features (X)

In [20]:
# Engineering HAR Features

df['rv_lag_1'] = df.groupby('Stock')['rv'].shift(1)
df['rv_rolling_5'] = df.groupby('Stock')['rv_lag_1'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['rv_rolling_22'] = df.groupby('Stock')['rv_lag_1'].rolling(window=22, min_periods=1).mean().reset_index(level=0, drop=True)

df['bpv_lag_1'] = df.groupby('Stock')['bpv'].shift(1)
df['bpv_rolling_5'] = df.groupby('Stock')['bpv_lag_1'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['good_lag_1'] = df.groupby('Stock')['good'].shift(1)
df['bad_lag_1'] = df.groupby('Stock')['bad'].shift(1)
df['bad_lag_5'] = df.groupby('Stock')['bad'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['rq_lag_1'] = df.groupby('Stock')['rq'].shift(1)
df['vix_lag_1'] = df.groupby('Stock')['vix'].shift(1)

epsilon = 1e-10
# df['bad_good_ratio_lag_1'] = df['bad_lag_1'] / (df['good_lag_1'] + epsilon)
# df['jump_ratio_lag_1'] = (df['rv_lag_1'] - df['bpv_lag_1']) / (df['rv_lag_1'] + epsilon)
# df['jump_ratio_lag_1'] = df['jump_ratio_lag_1'].clip(lower=0, upper=1)
df['rv_vix_interaction'] = df['rv_lag_1'] * df['vix_lag_1']
# df['rv_bpv_interaction'] = df['rv_lag_1'] * df['bpv_lag_1']
df['bad_rv_interaction'] = df['bad_lag_1'] * df['rv_lag_1']
# df['rv_rolling_std_1'] = df.groupby('Stock')['rv_lag_1'].shift(1).std()



df['jump_share'] = (df['rv'] - df['bpv']) / (df['rv'] + 1e-6)
df['jump_share'] = df['jump_share'].clip(lower=0)  
df['jump_share_rolling_5'] = (
    df.groupby('Stock')['jump_share']
      .rolling(5).mean()
      .reset_index(level=0, drop=True)
)

df['downside_share'] = df['bad'] / (df['good'] + df['bad'] + 1e-6)

df['kurtosis_proxy'] = df['rq'] / ((df['rv'] + 1e-6) ** 2)

df['vol_of_vol_10'] = (
    df.groupby('Stock')['rv']
      .rolling(window=10).std()
      .reset_index(level=0, drop=True)
)

df['vol_of_vol_5'] = (
    df.groupby('Stock')['rv']
      .rolling(window=5).std()
      .reset_index(level=0, drop=True)
)

df['vix_zscore_22'] = (
    df['vix'] -
    df.groupby('Stock')['vix'].rolling(22).mean().reset_index(level=0, drop=True)
) / (
    df.groupby('Stock')['vix'].rolling(22).std().reset_index(level=0, drop=True) + 1e-6
)


display(df.tail())

rv       bpv      good       bad        rq    vix  \
Date       Stock                                                            
2024-03-28 TRV    0.501900  0.450509  0.225921  0.275979  0.517683  13.01   
           UNH    0.774552  0.762729  0.406755  0.367797  0.696658  13.01   
           V      0.627872  0.445852  0.329977  0.297895  1.160977  13.01   
           VZ     0.783853  0.706211  0.493097  0.290755  1.847938  13.01   
           WMT    0.359440  0.336123  0.106481  0.252960  0.150975  13.01   

                  Y_reg  rv_lag_1  rv_rolling_5  rv_rolling_22  ...  \
Date       Stock                                                ...   
2024-03-28 TRV      NaN  0.460120      0.475846       0.647585  ...   
           UNH      NaN  0.520918      0.501865       1.205146  ...   
           V        NaN  0.736163      0.904239       0.672596  ...   
           VZ       NaN  1.289133      0.798543       1.050153  ...   
           WMT      NaN  0.405749      0.493407       0.576572  ...   

                  vix_lag_1  rv_vix_interaction  bad_rv_interaction  \
Date       Stock                                                      
2024-03-28 TRV        12.78            5.880331            0.063044   
           UNH        12.78            6.657337            0.151727   
           V          12.78            9.408158            0.328004   
           VZ         12.78           16.475119            0.291491   
           WMT        12.78            5.185475            0.086127   

                  jump_share  jump_share_rolling_5  downside_share  \
Date       Stock                                                     
2024-03-28 TRV      0.102394              0.060965        0.549867   
           UNH      0.015265              0.125783        0.474850   
           V        0.289899              0.158714        0.474451   
           VZ       0.099051              0.120061        0.370931   
           WMT      0.064871              0.032457        0.703758   

                  kurtosis_proxy  vol_of_vol_10  vol_of_vol_5  vix_zscore_22  
Date       Stock                                                              
2024-03-28 TRV          2.055076       0.218693      0.138913      -1.092715  
           UNH          1.161227       0.304659      0.187644      -1.092715  
           V            2.944961       0.436508      0.544517      -1.092715  
           VZ           3.007581       0.458418      0.350569      -1.092715  
           WMT          1.168558       0.111706      0.081198      -1.092715  

[5 rows x 26 columns]

In [18]:
df_model_ready = df.dropna()

print(f'Original panel size: {df.shape}')
print(f'Model-ready data size: {df_model_ready.shape}')

Original panel size: (160380, 26)
Model-ready data size: (152963, 26)


In [19]:
output_path = os.path.join('..', 'data', 'panel_data_model_ready.parquet')
df_model_ready.to_parquet(output_path)
display(df_model_ready.head())

rv       bpv      good       bad          rq    vix  \
Date       Stock                                                              
2003-02-03 AAPL   7.025066  6.011066  4.929262  2.095804  191.638996  31.02   
           AMGN   2.423086  2.491991  1.628248  0.794838    7.808419  31.02   
           AMZN   2.666827  2.511467  1.616915  1.049912   10.205894  31.02   
           AXP    3.101161  3.304581  1.450340  1.650822    9.342367  31.02   
           BA     5.870352  6.632632  3.171254  2.699098   50.703639  31.02   

                     Y_reg  rv_lag_1  rv_rolling_5  rv_rolling_22  ...  \
Date       Stock                                                   ...   
2003-02-03 AAPL   6.026879  8.196890      7.177777       6.109642  ...   
           AMGN   1.944900  3.797950      3.660728       3.129707  ...   
           AMZN   3.429612  5.460273      4.644653       6.378772  ...   
           AXP    4.478863  4.082929      6.028416       4.349161  ...   
           BA     4.191642  4.506228      6.684536       4.489795  ...   

                  vix_lag_1  rv_vix_interaction  bad_rv_interaction  \
Date       Stock                                                      
2003-02-03 AAPL       31.17          255.497050           20.739023   
           AMGN       31.17          118.382090            6.676749   
           AMZN       31.17          170.196696           15.206346   
           AXP        31.17          127.264895            7.118822   
           BA         31.17          140.459126            8.331612   

                  jump_share  jump_share_rolling_5  downside_share  \
Date       Stock                                                     
2003-02-03 AAPL     0.144340              0.186098        0.298332   
           AMGN     0.000000              0.145744        0.328027   
           AMZN     0.058256              0.082077        0.393693   
           AXP      0.000000              0.023774        0.532324   
           BA       0.000000              0.166030        0.459785   

                  kurtosis_proxy  vol_of_vol_10  vol_of_vol_5  vix_zscore_22  
Date       Stock                                                              
2003-02-03 AAPL         3.883139       3.177883      1.476383       1.055303  
           AMGN         1.329919       0.755096      0.813622       1.055303  
           AMZN         1.435031       4.191575      1.205634       1.055303  
           AXP          0.971422       2.706007      1.409176       1.055303  
           BA           1.471332       2.333607      2.772795       1.055303  

[5 rows x 26 columns]